In [ ]:
import torch

In [ ]:
N = 10

D_in = 1
D_out = 1

X = torch.randn(N, D_in)

true_W = torch.tensor([[2.0]])
true_b = torch.tensor(1.0)
y_true = X @ true_W + true_b + torch.randn(N, D_out) * 0.1

In [ ]:
W = torch.randn(D_in, D_out, requires_grad=True)
b = torch.randn(1, requires_grad=True)

print(f"Initial Weight W:\n {W}\n")
print(f"Initial Weight b:\n {b}")

Initial Weight W:
 tensor([[1.1812]], requires_grad=True)

Initial Weight b:
 tensor([0.7234], requires_grad=True)


In [ ]:
y_hat = X @ W + b

print(f"Prediction y_hat (first 3 rows):\n {y_hat[:3]}\n")
print(f"True y (first 3 rows):\n {y_true[:3]}")

# Our Prediction is way out of the real.. so we have to do a backward pass

Prediction y_hat (first 3 rows):
 tensor([[0.6015],
        [1.6534],
        [2.5662]], grad_fn=<SliceBackward0>)

True y (first 3 rows):
 tensor([[0.8752],
        [2.5863],
        [4.1202]])


In [ ]:
# Mean Squared Error
def MSE (y_true, Y_pred):
  error = Y_pred - y_true
  squared_error = error ** 2
  loss = squared_error.mean()
  return loss

loss = MSE(y_true, y_hat)
print(f"Loss: {loss}")

Loss: 0.7246080040931702


In [ ]:
# Compute gradients
loss.backward()

# The gradients are now stored in the .grad attribute
print(f"Gradient for W: {W.grad}\n")
print(f"Gradient for b: {b.grad}")

# Sign is everything, Negative gradient -> increasing will decrease loss while Positive gradient -> reducing will decrease loss... so just go in opposite direction of the gradient.

Gradient for W: tensor([[-1.4771]])

Gradient for b: tensor([-0.7668])


Learning With Gradient Descent

$$\theta_{t+1} = \theta_t - \eta .\nabla_\theta L$$

θ(theta) represents all our parameters. For us, that's W and b.

η (eta) is the learning rate. A small number (e.g., 0.01) controlling our step size.

$\nabla_\theta L$ is the gradient of the loss, We just calculated this!. it's in W.grad and b.grad.
_________________________________________

W_new = W_old - learning_rate * W.grad

b_new = b_old - learning_rate * b_grad

In [ ]:
# The Entire Training Process

# Hyperparameters
learning_rate, epochs = 0.01, 100

# Re-initialize parameters
W, b = torch.randn(1, 1, requires_grad=True), torch.randn(1, requires_grad=True)

# Training Loop

for epoch in range(epochs):
  # Forward pass and loss
  y_hat = X @ W + b
  loss = torch.mean((y_hat - y_true)**2)
  #  Backward Pass
  loss.backward()

  # Update parameters
  with torch.no_grad():
    W -= learning_rate * W.grad
    b -= learning_rate * b.grad

  # rest gradients to zero
  W.grad.zero_()
  b.grad.zero_()

  if epoch % 10 == 0:
    print(f"Epoch {epoch:02d}: Loss={loss.item():.4f}, W={W.item():.3f}, b={b.item():.3f}")

print(f"\nFinal Parameters: W={W.item():.3f}, b={b.item():.3f}")
print(f"True Parameters: W=2.000, b=1.000")

Epoch 00: Loss=4.9159, W=-0.039, b=0.160
Epoch 10: Loss=3.3044, W=0.300, b=0.362
Epoch 20: Loss=2.2313, W=0.581, b=0.519
Epoch 30: Loss=1.5141, W=0.815, b=0.640
Epoch 40: Loss=1.0327, W=1.010, b=0.733
Epoch 50: Loss=0.7083, W=1.172, b=0.804
Epoch 60: Loss=0.4887, W=1.308, b=0.858
Epoch 70: Loss=0.3394, W=1.421, b=0.898
Epoch 80: Loss=0.2374, W=1.516, b=0.929
Epoch 90: Loss=0.1673, W=1.595, b=0.951

Final Parameters: W=1.656, b=0.966
True Parameters: W=2.000, b=1.000


**Using torch.nn MODULE**

Note:
- A parameter is a special Tensor that requires_grad=True by default, Auto registers with the model, handles all bookkeeping

- nn.ReLU() is an activation function that turns any input that is negative to zero.  Basically to introduce Non-linearities between the linear layers. just nn.Linear cannot handle complex patterns.. just simple ones and real life is complex and rarely simple. it powers Classic CNNs and basic feed-forward networks

- nn.GELU() - GAUSSIAN ERROR LINEAR UNIT is the Modern Standard for Transformers (GPT, Llama, BERT). A smoother, gently curving version of ReLU.

- ReLU introduces sharp corners.. while GELU has Smooth, continuous S-curve.

- nn.SOFTMAX is used on final output layer for classification problems. its job is to convert the raw model scores (logits) into a probability distribution (output will between 0 and 1 and all add up to 1).

- nn.EMBEDDING converts Words to Numbers, it is a giant Learnable lookup table (Each word gets its own unique vector)

- nn.LAYERNORM Prevents values from exploding/vanishing,  it rescales everything to a stable range. In deep neural networks, numbers can sometimes get wildly large or tiny as they pass through layers. This makes the model "explode" or "vanish," causing it to stop learning. nn.LayerNorm forces the numbers to stay within a healthy, predictable range.

- nn.DROPOUT prevents overfitting, during training it randomly zeros out some of the neurons,  it forces the network to be robust and not rely on any single neuron. it only happens during training; Dropout randomly "turns off" neurons during training to stop the model from memorizing specific paths.
It forces the network to work as a team, making it robust and better at handling new, unseen data.

- nn.Module is the skeleton of your model that holds all the layers and parameters in one organized place.  it is the base class for all neural network objects in PyTorch. When you create a model, you "inherit" from it. It acts like a container that tracks everything your model contains.

- torch.optim is the engine that uses the gradients to update your weights and make the model smarter after every mistake. Once the Blueprint (nn.Module) makes a prediction and the Loss Function calculates how wrong that prediction was, the Optimizer (torch.optim) steps in to fix it.


In [ ]:
#  The input has one feature, the outpuut has one value
D_in = 1
D_out = 1

# Create a linear layer
linear_layer = torch.nn.Linear(in_features=D_in, out_features=D_out)

print(f"Layer's Weight (W): {linear_layer.weight}\n")
print(f"Layer's Bias (b): {linear_layer.bias}\n")

# You can use it just like a func. Thie is the forward pass.
# (Assume X is a tensor of shape [10, 1] from previous chapters)
y_hat_nn = linear_layer(X)

print(f"Output of nn.linear (first 3 rows):\n {y_hat_nn[:3]}")


Layer's Weight (W): Parameter containing:
tensor([[-0.1597]], requires_grad=True)

Layer's Bias (b): Parameter containing:
tensor([-0.0978], requires_grad=True)

Output of nn.linear (first 3 rows):
 tensor([[-0.0814],
        [-0.2236],
        [-0.3470]], grad_fn=<SliceBackward0>)


In [ ]:
import torch.nn as nn

# Inherit from nn.module
class LinearRegressionModel(nn.Module):
  def __init__(self, in_features, out_features):
    super().__init__()
    # In the Constructor, we define the layers we'll use.
    self.linear_layer = nn.Linear(in_features, out_features)

  def forward(self, x):
    # In the forward pass, we CONNECT the layers.
    return self.linear_layer(x)

# Instantiate the model
model = LinearRegressionModel(in_features=1, out_features=1)
print("Model Architecture")
print(model)

Model Architecture
LinearRegressionModel(
  (linear_layer): Linear(in_features=1, out_features=1, bias=True)
)


In [ ]:
import torch.optim as optim

# Hyperparameters
learning_rate = 0.01

# create an Adam optimizer.
# We pass model.parameters() to tell it which tensors to manage.
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# We'll also grab a prebuilt loss function from torch.nn
loss_fn = nn.MSELoss() # Mean Squared Error Loss

In [ ]:
# Clean Training Loop
epochs = 100

for epoch in range(epochs):
  ### FORWARD PASS ###
  y_hat = model(X)

  ### CALCULATE LOSS ###
  loss = loss_fn(y_hat, y_true)

  ### THE THREE-LINE MANTRA RULE ###
  # 1. Zero the gradients
  optimizer.zero_grad()
  # 2. Compute gradients
  loss.backward()
  # 3. Update the parameters
  optimizer.step()

  # Print progress
  if epoch % 10 == 0:
    print(f"Epoch {epoch:02d}: Loss={loss.item():.4f}")

Epoch 00: Loss=0.0137
Epoch 10: Loss=0.0123
Epoch 20: Loss=0.0111
Epoch 30: Loss=0.0100
Epoch 40: Loss=0.0092
Epoch 50: Loss=0.0085
Epoch 60: Loss=0.0080
Epoch 70: Loss=0.0075
Epoch 80: Loss=0.0071
Epoch 90: Loss=0.0068
